# Policy-academic alignment analysis

Builds country-level field distribution vectors from the classified OpenAlex (academic) and Overton (policy) corpora, computes cosine similarity and Pearson r as alignment metrics, and produces the heatmap, radar chart, and alignment score figures used in the paper.

**Inputs:**
- `training_dataset.csv` (OpenAlex publications, labeled)
- `policy_classified.jsonl` (Overton documents with predicted field, from `3_policy_classification.ipynb`)

**Outputs:**
- `heatmap_academic_vs_policy.png`
- `radar_academic_vs_policy.png`
- `alignment_scores.png`
- `country_field_distributions.csv` (academic and policy distributions per country)

## Setup

In [ ]:
%pip install matplotlib seaborn numpy pandas scipy --quiet

In [ ]:
import json
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Colab Notebooks"

## 1. Load and build field distributions

In [ ]:
df = pd.read_csv(f"{BASE}/training_dataset.csv")
all_labels = sorted(df['label'].unique())

academic = {}
for code in df['country'].unique():
    counts = Counter(dict(df[df['country'] == code]['label'].value_counts()))
    total = sum(counts.values())
    academic[code] = {l: counts.get(l, 0) / total for l in all_labels}

In [ ]:
policy_counts = defaultdict(Counter)
with open(f"{BASE}/policy_classified.jsonl") as f:
    for line in f:
        doc = json.loads(line)
        code = doc.get("asean_code") or "INTL"
        label = doc.get("pred_label")
        if label:
            policy_counts[code][label] += 1

policy = {}
for code, counts in policy_counts.items():
    total = sum(counts.values())
    policy[code] = {l: counts.get(l, 0) / total for l in all_labels}

In [ ]:
# Countries present in both corpora, excluding non-ASEAN international sources
countries = sorted([c for c in academic if c in policy and c != 'INTL'])
print(f"countries analyzed: {countries}")

## 2. Heatmap: academic vs policy distribution

In [ ]:
LABEL_SHORT = {
    'Agricultural and Biological Sciences': 'Agri & Bio',
    'Biochemistry, Genetics and Molecular Biology': 'Biochem',
    'Chemical Engineering': 'Chem Eng',
    'Chemistry': 'Chemistry',
    'Computer Science': 'CS',
    'Decision Sciences': 'Decision Sci',
    'Dentistry': 'Dentistry',
    'Earth and Planetary Sciences': 'Earth Sci',
    'Energy': 'Energy',
    'Engineering': 'Engineering',
    'Environmental Science': 'Environ Sci',
    'Health Professions': 'Health Prof',
    'Immunology and Microbiology': 'Immunology',
    'Materials Science': 'Materials',
    'Mathematics': 'Math',
    'Medicine': 'Medicine',
    'Neuroscience': 'Neurosci',
    'Nursing': 'Nursing',
    'Pharmacology, Toxicology and Pharmaceutics': 'Pharma',
    'Physics and Astronomy': 'Physics',
    'Veterinary': 'Veterinary',
}
short_labels = [LABEL_SHORT[l] for l in all_labels]

In [ ]:
acad_mat = np.array([[academic[c].get(l, 0) for l in all_labels] for c in countries])
poli_mat = np.array([[policy[c].get(l, 0) for l in all_labels] for c in countries])

fig, axes = plt.subplots(2, 1, figsize=(16, 12))
for ax, mat, title in zip(
    axes,
    [acad_mat, poli_mat],
    ['Academic publication distribution (OpenAlex)', 'Policy document distribution (Overton)'],
):
    sns.heatmap(
        mat * 100,
        ax=ax,
        xticklabels=short_labels,
        yticklabels=countries,
        cmap='YlOrRd',
        annot=True, fmt='.1f',
        annot_kws={'size': 7},
        cbar_kws={'label': '%'},
        linewidths=0.3,
    )
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.suptitle('ASEAN S&T field distribution: academic vs policy', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f"{BASE}/heatmap_academic_vs_policy.png", dpi=150, bbox_inches='tight')
plt.show()

## 3. Radar charts by country

In [ ]:
def radar_chart(ax, academic_vals, policy_vals, labels, title):
    n = len(labels)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles += angles[:1]

    a_vals = list(academic_vals) + [academic_vals[0]]
    p_vals = list(policy_vals) + [policy_vals[0]]

    ax.plot(angles, a_vals, 'o-', color='steelblue', linewidth=1.5, label='Academic')
    ax.fill(angles, a_vals, alpha=0.15, color='steelblue')
    ax.plot(angles, p_vals, 's-', color='tomato', linewidth=1.5, label='Policy')
    ax.fill(angles, p_vals, alpha=0.15, color='tomato')

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, size=6)
    ax.set_title(title, size=10, fontweight='bold', pad=15)
    ax.set_ylim(0, None)
    ax.grid(True, alpha=0.3)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 15), subplot_kw=dict(polar=True))
axes_flat = axes.flatten()

for i, code in enumerate(countries):
    a_vals = np.array([academic[code].get(l, 0) for l in all_labels]) * 100
    p_vals = np.array([policy[code].get(l, 0) for l in all_labels]) * 100
    cos_sim = 1 - cosine(a_vals, p_vals)
    radar_chart(axes_flat[i], a_vals, p_vals, short_labels, title=f"{code} (cosine={cos_sim:.3f})")

for j in range(len(countries), len(axes_flat)):
    axes_flat[j].set_visible(False)

legend_handles = [
    mpatches.Patch(color='steelblue', alpha=0.5, label='Academic (OpenAlex)'),
    mpatches.Patch(color='tomato', alpha=0.5, label='Policy (Overton)'),
]
fig.legend(handles=legend_handles, loc='lower right', fontsize=11, bbox_to_anchor=(0.98, 0.02))

plt.suptitle('ASEAN S&T field distribution: academic vs policy (radar charts)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{BASE}/radar_academic_vs_policy.png", dpi=150, bbox_inches='tight')
plt.show()

## 4. Alignment scores

Cosine similarity captures overall directional similarity between the two distributions; Pearson r captures field-by-field proportional correspondence.

In [ ]:
cosine_scores = []
pearson_scores = []

for code in countries:
    a = np.array([academic[code].get(l, 0) for l in all_labels])
    p = np.array([policy[code].get(l, 0) for l in all_labels])
    cosine_scores.append(1 - cosine(a, p))
    pearson_scores.append(pearsonr(a, p)[0])

alignment_df = pd.DataFrame({
    'country': countries,
    'cosine_similarity': cosine_scores,
    'pearson_r': pearson_scores,
}).sort_values('cosine_similarity', ascending=False)
alignment_df

In [ ]:
x = np.arange(len(countries))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width / 2, cosine_scores, width, label='Cosine similarity', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width / 2, pearson_scores, width, label='Pearson r', color='tomato', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(countries, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Policy-academic alignment by country (ASEAN)', fontsize=13, fontweight='bold')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='threshold = 0.5')
ax.legend(fontsize=10)
ax.set_ylim(0, 1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f"{BASE}/alignment_scores.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. Export country-field distributions

Saves the academic and policy distributions in long format for reuse (e.g. as `data/country_field_distributions.csv` in the replication repo).

In [ ]:
rows = []
for code in countries:
    for label in all_labels:
        rows.append({
            'country': code,
            'field': label,
            'academic_share': academic[code].get(label, 0),
            'policy_share': policy[code].get(label, 0),
        })

distributions_df = pd.DataFrame(rows)
distributions_df.to_csv(f"{BASE}/country_field_distributions.csv", index=False)
distributions_df.head()